# Intermittent Demand: Croston, SBA and SBJ

Intermittent demand has many periods of zero. [Croston (1972)](https://doi.org/10.1057/jors.1972.50) proposed smoothing the demand sizes and the intervals between demands separately, and forecasting their ratio. This tutorial classifies a simulated series, fits the three methods, and compares each forecast with the true rate of demand, which simulation lets us know.

In [ ]:
Pollis.packages("Durbyn", "Plots")

In [ ]:
using Durbyn, Random, Statistics
import Plots

Random.seed!(11)

## Simulating Demand with a Known Rate

In each period a demand occurs with probability 0.25, and its size is a whole number from 1 to 9. The true rate is therefore 0.25 × 5 = 1.25 units per period.

In [ ]:
n = 300
prob, mean_size = 0.25, 5
y = [rand() < prob ? Float64(rand(1:9)) : 0.0 for _ in 1:n]
true_rate = prob * mean_size
train, test = y[1:end-60], y[end-59:end]
Plots.bar(y, label="demand", xlabel="period", title="Intermittent demand", linecolor=:match)

## Classifying the Pattern

ADI is the average interval between demands, and CV2 is the squared coefficient of variation of the demand sizes. The cut-offs of 1.32 and 0.49 separate smooth, erratic, intermittent and lumpy demand ([Syntetos, Boylan and Croston (2005)](https://doi.org/10.1057/palgrave.jors.2601841)).

In [ ]:
sizes = train[train .> 0]
adi = length(train) / length(sizes)
cv2 = (std(sizes) / mean(sizes))^2
pattern = adi < 1.32 ? (cv2 < 0.49 ? "smooth" : "erratic") : (cv2 < 0.49 ? "intermittent" : "lumpy")
println("ADI ", round(adi, digits=2), ", CV2 ", round(cv2, digits=2), ": ", pattern)

## Fitting the Three Methods

`croston_classic` forecasts the ratio of the smoothed size to the smoothed interval, which is too high on average. `croston_sba` multiplies the ratio by 1 − a/2, with a the smoothing parameter of the interval ([Syntetos and Boylan (2005)](https://doi.org/10.1016/j.ijforecast.2004.10.001)), and `croston_sbj` by 1 − a/(2 − a) ([Shale, Boylan and Johnston (2006)](https://doi.org/10.1057/palgrave.jors.2602031)).

In [ ]:
methods = (("Classic", croston_classic), ("SBA", croston_sba), ("SBJ", croston_sbj))
for (name, method) in methods
    model = method(train)
    rate = forecast(model; h=60).mean[1]
    println(rpad(name, 9), "weights ", round.(model.weights, digits=3), "  rate ", round(rate, digits=3), "  true rate ", true_rate)
end

## Reading the Forecast as a Total

The forecast is one flat rate, so a single period's demand is not what it predicts. The useful comparison is the total over the 60 periods held back: rate × 60 against the demand observed.

In [ ]:
println("Observed total ", sum(test))
for (name, method) in methods
    rate = forecast(method(train); h=60).mean[1]
    println(rpad(name, 9), "forecast total ", round(rate * 60, digits=1))
end

At the true rate the expected total is 75. Demand over 60 periods varies widely around that figure, so a gap between the forecast and the observed total can come from chance alone, and one holdout cannot rank the three methods.

In [ ]:
sba_rate = forecast(croston_sba(train); h=60).mean
Plots.plot(cumsum(test), label="observed", xlabel="period held back", ylabel="cumulative demand", title="Cumulative demand")
Plots.plot!(cumsum(sba_rate), label="SBA", lw=2)

## Bias over Many Simulated Series

One series cannot show a bias, so the fit is repeated on many simulated series. The smoothing parameters are small here, and the corrections are small with them, so expect the three means to stay close to one another.

In [ ]:
reps = 100
rates = Dict(name => Float64[] for (name, _) in methods)
for _ in 1:reps
    z = [rand() < prob ? Float64(rand(1:9)) : 0.0 for _ in 1:120]
    for (name, method) in methods
        push!(rates[name], forecast(method(z); h=1).mean[1])
    end
end
for (name, _) in methods
    println(rpad(name, 9), "mean rate ", round(mean(rates[name]), digits=3), "  true rate ", true_rate)
end

These methods return no prediction interval, as `fc` holds only `mean`. A safety stock needs a separate assumption on the distribution of demand over the lead time.

## References

Croston, J. D. (1972). Forecasting and stock control for intermittent demands. *Operational Research Quarterly* 23(3), 289–303. [doi:10.1057/jors.1972.50](https://doi.org/10.1057/jors.1972.50)

Shale, E. A., Boylan, J. E. and Johnston, F. R. (2006). Forecasting for intermittent demand: the estimation of an unbiased average. *Journal of the Operational Research Society* 57(5), 588–592. [doi:10.1057/palgrave.jors.2602031](https://doi.org/10.1057/palgrave.jors.2602031)

Syntetos, A. A. and Boylan, J. E. (2005). The accuracy of intermittent demand estimates. *International Journal of Forecasting* 21(2), 303–314. [doi:10.1016/j.ijforecast.2004.10.001](https://doi.org/10.1016/j.ijforecast.2004.10.001)

Syntetos, A. A., Boylan, J. E. and Croston, J. D. (2005). On the categorization of demand patterns. *Journal of the Operational Research Society* 56(5), 495–503. [doi:10.1057/palgrave.jors.2601841](https://doi.org/10.1057/palgrave.jors.2601841)